In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import os
from dotenv import load_dotenv
import requests
import json
import pandas as pd
import time

In [2]:
#load_dotenv()
API_KEY = os.getenv('API_KEY')
print(API_KEY)

bbc56ce0260a4036ba8750315199a7d1


In [ ]:

# API_KEY = os.getenv('API_KEY')
# BLS_V2_URL = "https://api.bls.gov/publicAPI/v2/timeseries/data/"  


# state_fips = [
#     '01','02','04','05','06','08','09','10','12','13',
#     '15','16','17','18','19','20','21','22','23','24',
#     '25','26','27','28','29','30','31','32','33','34',
#     '35','36','37','38','39','40','41','42','44','45',
#     '46','47','48','49','50','51','53','54','55','56'
# ]
# state_names = [
#     'Alabama','Alaska','Arizona','Arkansas','California','Colorado',
#     'Connecticut','Delaware','Florida','Georgia','Hawaii','Idaho',
#     'Illinois','Indiana','Iowa','Kansas','Kentucky','Louisiana',
#     'Maine','Maryland','Massachusetts','Michigan','Minnesota',
#     'Mississippi','Missouri','Montana','Nebraska','Nevada',
#     'New Hampshire','New Jersey','New Mexico','New York',
#     'North Carolina','North Dakota','Ohio','Oklahoma','Oregon',
#     'Pennsylvania','Rhode Island','South Carolina','South Dakota',
#     'Tennessee','Texas','Utah','Vermont','Virginia','Washington',
#     'West Virginia','Wisconsin','Wyoming'
# ]

# # saved_state = [    
# #     'Alabama',  'California',    'Delaware',     'Florida',       'Idaho',
# #     'Iowa',      'Kansas',       'Maine',    'Michigan',   'Minnesota',
# #     'Puerto Rico'
# # ]

# # for name in state_names:
# #     for name2 in saved_state:
# #         if name == name2:
# #             new_list= 



# sectors = {
#     '06': 'Goods_Producing',
#     '07': 'Service_Providing',
#     '10': 'Mining_and_Logging',
#     '20': 'Construction',
#     '30': 'Manufacturing',
#     '40': 'Trade_Transport_and_Utilities',
#     '50': 'Information',
#     '55': 'Financial_Activities',
#     '60': 'Professional_and_Tech_Services',
#     '65': 'Education_and_Health_Services',
#     '70': 'Leisure_and_Hospitality',
#     '90': 'Government',
# }

# # v2 limits: 50 series/request, 20 years/request
# # 2016-2026 fits in one window; split if you ever go beyond 20 years
# TIME_PERIODS  = [("2016", "2026")]
# BATCH_SIZE    = 50




# def chunked(lst: list, size: int):
#     for i in range(0, len(lst), size):
#         yield lst[i:i + size]


# def safe_float(value: str):
#     try:
#         return float(value)
#     except (ValueError, TypeError):
#         return None

# # ── PASTE THIS VERIFICATION BLOCK BEFORE THE FETCH LOOP ──────────────────────
# print("=== SERIES ID VERIFICATION ===")
# expected = {
#     '06': 'SMU01000000060000001',   # Goods Producing
#     '07': 'SMU01000000070000001',   # Service Providing
#     '10': 'SMU01000000100000001',   # Mining & Logging
#     '20': 'SMU01000000200000001',   # Construction
#     '30': 'SMU01000000300000001',   # Manufacturing
#     '40': 'SMU01000000400000001',   # Trade Transport
#     '50': 'SMU01000000500000001',   # Information  ← was wrong before
#     '55': 'SMU01000000550000001',   # Financial
#     '60': 'SMU01000000600000001',   # Prof & Tech  ← was wrong before
#     '65': 'SMU01000000650000001',   # Education
#     '70': 'SMU01000000700000001',   # Leisure      ← was wrong before
#     '90': 'SMU01000000900000001',   # Government   ← was wrong before
# }
# all_ok = True
# for code, exp in expected.items():
#     gen = f"SMU01000000{code.zfill(3)}0000001"  # wait — let's check
#     # '50'.zfill(3) = '050' not '500'!
#     # We need a different approach for codes >= 10
#     gen = f"SMU01" + "00000" + code.zfill(3) + "0000001"
#     ok  = gen == exp
#     print(f"  {'' if ok else ''} {code} → {gen} {'==' if ok else '!='} {exp}")
#     all_ok = False if not ok else all_ok

# if not all_ok:
#     raise ValueError(" Fix series IDs before fetching!")
# else:
#     print("   All IDs correct!\n")

# def fetch_bls_v2(series_batch: list[str], start_year: str, end_year: str) -> dict:
#     """POST a batch of series to the BLS v2 API and return parsed JSON."""
#     payload = {
#         "seriesid":        series_batch,
#         "startyear":       start_year,       
#         "endyear":         end_year,          
#         "registrationkey": API_KEY,           
#         "catalog":         False,
#         "calculations":    False,
#         "annualaverage":   False,             
#     }
#     headers  = {"Content-type": "application/json"}
#     response = requests.post(BLS_V2_URL, data=json.dumps(payload), headers=headers)  
#     response.raise_for_status()
#     return response.json()


# def parse_bls_response(json_data: dict, id_to_meta: dict) -> list[dict]:
#     """Extract monthly records from a BLS API response."""
#     records = []
#     status  = json_data.get("status")
#     if status != "REQUEST_SUCCEEDED":
#         print(f"  API warning ({status}): {json_data.get('message', 'Unknown error')}")
#         # if "Results" not in json_data:
#         #     return records  

#     for series in json_data["Results"]["series"]:
#         sid  = series["seriesID"]
#         meta = id_to_meta.get(sid, {})
#         for item in series["data"]:
#             period = item["period"]          
#             if "M01" <= period <= "M12":
#                 records.append({
#                     **meta,
#                     "Year":  int(item["year"]),
#                     "Month": int(period.replace("M", "")),
#                     "Value": safe_float(item["value"]),
#                 })
#     return records


# # STEP 1: STATE UNEMPLOYMENT (LAUS) 
# print("=" * 60)
# print("STEP 1: Fetching state unemployment rates (LAUS)...")
# print("=" * 60)

# laus_series = [f"LASST{fips}0000000000003" for fips in state_fips]
# laus_meta   = {
#     f"LASST{fips}0000000000003": {"State": name, "Metric": "Unemployment_Rate"}
#     for fips, name in zip(state_fips, state_names)   
# }

# laus_records = []
# for start, end in TIME_PERIODS:
#     for batch in chunked(laus_series, BATCH_SIZE):
#         print(f"  LAUS: {len(batch)} states | {start}–{end}")
#         result = fetch_bls_v2(batch, start, end)
#         laus_records.extend(parse_bls_response(result, laus_meta))
#         time.sleep(0.5)

# df_laus = pd.DataFrame(laus_records)[["State", "Year", "Month", "Value"]]
# df_laus = df_laus.rename(columns={"Value": "Unemployment_Rate"})
# print(f"  ✓ {len(df_laus):,} LAUS records fetched")


# # STEP 2: SECTOR EMPLOYMENT (SMU) 
# print("\n" + "=" * 60)
# print("STEP 2: Fetching sector employment (SMU)...")
# print(f"  {len(state_fips)} states × {len(sectors)} sectors = {len(state_fips)*len(sectors)} series")
# print("=" * 60)

# smu_series = []
# smu_meta   = {}
# for fips, state_name in zip(state_fips, state_names):
#     for sector_code, sector_name in sectors.items():
#         sid = f"SMU{fips}000000{sector_code}0000001"
#         smu_series.append(sid)
#         smu_meta[sid] = {"State": state_name, "Sector": sector_name}

# smu_records  = []
# total_batches = len(list(chunked(smu_series, BATCH_SIZE))) * len(TIME_PERIODS)
# batch_count   = 0

# for start, end in TIME_PERIODS:
#     for batch in chunked(smu_series, BATCH_SIZE):
#         batch_count += 1
#         print(f"  Batch {batch_count}/{total_batches} | {len(batch)} series | {start}–{end}")
#         result = fetch_bls_v2(batch, start, end)
#         smu_records.extend(parse_bls_response(result, smu_meta))
#         time.sleep(0.5)

# df_smu = pd.DataFrame(smu_records)
# print(f"  ✓ {len(df_smu):,} SMU records fetched")


# #  STEP 3: PIVOT SECTORS INTO COLUMNS 
# print("\n" + "=" * 60)
# print("STEP 3: Pivoting sectors into columns...")
# print("=" * 60)

# df_sectors = df_smu.pivot_table(
#     index   = ["State", "Year", "Month"],
#     columns = "Sector",
#     values  = "Value",
#     aggfunc = "first",
# ).reset_index()
# df_sectors.columns.name = None


# #  STEP 4: MERGE 
# print("\n" + "=" * 60)
# print("STEP 4: Merging unemployment + sectors...")
# print("=" * 60)

# df_merged = df_laus.merge(df_sectors, on=["State", "Year", "Month"], how="left")

# df_merged["Date"] = pd.to_datetime(
#     df_merged[["Year", "Month"]].rename(
#         columns={"Year": "year", "Month": "month"}
#     ).assign(day=1)
# )

# df_merged["Total_Tech_Employment_Thousands"] = (
#     df_merged.get("Information",                    pd.Series(0, index=df_merged.index)).fillna(0) +
#     df_merged.get("Professional_and_Tech_Services", pd.Series(0, index=df_merged.index)).fillna(0)
# )

# sector_cols = list(sectors.values())
# col_order   = (
#     ["State", "Date", "Year", "Month", "Unemployment_Rate"]
#     + [c for c in sector_cols if c in df_merged.columns]
#     + ["Total_Tech_Employment_Thousands"]
# )
# df_merged = df_merged[col_order].sort_values(["State", "Year", "Month"]).reset_index(drop=True)


# #  STEP 5: SUMMARY & EXPORT 
# print(f"\n{'=' * 60}")
# print("FINAL SUMMARY")
# print(f"{'=' * 60}")
# print(f"  Total records  : {len(df_merged):,}")
# print(f"  States         : {df_merged['State'].nunique()}")
# print(f"  Sectors        : {len(sectors)}")
# print(f"  Date range     : {df_merged['Date'].min().strftime('%b %Y')} → {df_merged['Date'].max().strftime('%b %Y')}")
# print(f"\nMissing values per column:")
# print(df_merged.isna().sum().to_string())
# print(f"\nSample (first 5 rows):")
# print(df_merged.head())

# out_file = "state_unemployment_all_sectors_2016_2026_v2.csv"
# df_merged.to_csv(out_file, index=False)
# print(f"\n✓ Saved: {out_file}")

In [ ]:
import os, requests, json, time
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv('API_KEY')
print(f"✓ API Key loaded: {len(API_KEY) if API_KEY else 0} chars")

BLS_V2_URL = "https://api.bls.gov/publicAPI/v2/timeseries/data/"

state_fips = [
    '01','02','04','05','06','08','09','10','12','13',
    '15','16','17','18','19','20','21','22','23','24',
    '25','26','27','28','29','30','31','32','33','34',
    '35','36','37','38','39','40','41','42','44','45',
    '46','47','48','49','50','51','53','54','55','56'
]
state_names = [
    'Alabama','Alaska','Arizona','Arkansas','California','Colorado',
    'Connecticut','Delaware','Florida','Georgia','Hawaii','Idaho',
    'Illinois','Indiana','Iowa','Kansas','Kentucky','Louisiana',
    'Maine','Maryland','Massachusetts','Michigan','Minnesota',
    'Mississippi','Missouri','Montana','Nebraska','Nevada',
    'New Hampshire','New Jersey','New Mexico','New York',
    'North Carolina','North Dakota','Ohio','Oklahoma','Oregon',
    'Pennsylvania','Rhode Island','South Carolina','South Dakota',
    'Tennessee','Texas','Utah','Vermont','Virginia','Washington',
    'West Virginia','Wisconsin','Wyoming'
]

# 12 sectors — 2-digit keys (trailing '0' appended in series ID builder)
sectors = {
    '06': 'Goods_Producing',
    '07': 'Service_Providing',
    '10': 'Mining_and_Logging',
    '20': 'Construction',
    '30': 'Manufacturing',
    '40': 'Trade_Transport_and_Utilities',
    '50': 'Information',
    '55': 'Financial_Activities',
    '60': 'Professional_and_Tech_Services',
    '65': 'Education_and_Health_Services',
    '70': 'Leisure_and_Hospitality',
    '90': 'Government',
}

TIME_PERIODS = [("2016", "2026")]
BATCH_SIZE   = 50

# ── VERIFY SERIES ID FORMAT BEFORE FETCHING ───────────────────────────────────
print("\n=== SERIES ID VERIFICATION ===")
expected_ids = {
    '06': 'SMU01000000600000001',
    '07': 'SMU01000000700000001',
    '10': 'SMU01000001000000001',
    '20': 'SMU01000002000000001',
    '30': 'SMU01000003000000001',
    '40': 'SMU01000004000000001',
    '50': 'SMU01000005000000001',
    '55': 'SMU01000005500000001',
    '60': 'SMU01000006000000001',
    '65': 'SMU01000006500000001',
    '70': 'SMU01000007000000001',
    '90': 'SMU01000009000000001',
}
all_ok = True
for code, expected in expected_ids.items():
    three = code + '0'                          # '06'→'060', '50'→'500'
    generated = f"SMU01000000{code}0000001"     # wrong — just for display
    generated = f"SMU01" + "00000" + three + "0000001"
    ok = generated == expected
    print(f"  {'ok' if ok else 'x'} {sectors[code]:35s}: {generated}")
    if not ok:
        all_ok = False

if not all_ok:
    raise ValueError("Series ID format mismatch — do not proceed!")
print("   All 12 series IDs verified!\n")

# ── HELPERS ───────────────────────────────────────────────────────────────────

def chunked(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

def safe_float(val):
    try:
        return float(val)
    except (ValueError, TypeError):
        return None

def fetch_bls_v2(series_batch, start_year, end_year, max_attempts=5):
    payload = {
        "seriesid":        series_batch,
        "startyear":       start_year,
        "endyear":         end_year,
        "registrationkey": API_KEY,
        "catalog":         False,
        "calculations":    False,
        "annualaverage":   False,
    }
    headers = {"Content-type": "application/json"}
    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.post(
                BLS_V2_URL, data=json.dumps(payload),
                headers=headers, timeout=(10, 60)
            )
            response.raise_for_status()
            return response.json()
        except requests.exceptions.ReadTimeout:
            wait = 2 ** attempt
            print(f"  Timeout attempt {attempt}/{max_attempts}. Retrying in {wait}s...")
            time.sleep(wait)
        except Exception as e:
            print(f"  Error: {e}. Retrying in 5s...")
            time.sleep(5)
    print("  ✗ Batch failed. Skipping.")
    return None

def parse_bls_response(json_data, id_to_meta):
    records = []
    if json_data is None:
        return records
    status = json_data.get("status")
    if status != "REQUEST_SUCCEEDED":
        print(f"  API warning ({status}): {json_data.get('message', 'Unknown error')}")
        return records
    for series in json_data["Results"]["series"]:
        sid  = series["seriesID"]
        meta = id_to_meta.get(sid, {})
        for item in series["data"]:
            period = item["period"]
            if "M01" <= period <= "M12":
                records.append({
                    **meta,
                    "Year":  int(item["year"]),
                    "Month": int(period.replace("M", "")),
                    "Value": safe_float(item["value"]),
                })
    return records

# ── STEP 1: UNEMPLOYMENT (LAUS) ───────────────────────────────────────────────

print("=" * 60)
print("STEP 1: Fetching state unemployment rates (LAUS)...")
print("=" * 60)

laus_series = [f"LASST{fips}0000000000003" for fips in state_fips]
laus_meta   = {
    f"LASST{fips}0000000000003": {"State": name}
    for fips, name in zip(state_fips, state_names)
}

laus_records = []
for start, end in TIME_PERIODS:
    for batch in chunked(laus_series, BATCH_SIZE):
        print(f"  LAUS: {len(batch)} states | {start}–{end}")
        result = fetch_bls_v2(batch, start, end)
        laus_records.extend(parse_bls_response(result, laus_meta))
        time.sleep(1)

df_laus = pd.DataFrame(laus_records)[["State", "Year", "Month", "Value"]]
df_laus = df_laus.rename(columns={"Value": "Unemployment_Rate"})
print(f"  ✓ {len(df_laus):,} LAUS records fetched")

# ── STEP 2: SECTOR EMPLOYMENT (SMU) ───────────────────────────────────────────

print("\n" + "=" * 60)
print("STEP 2: Fetching sector employment (SMU)...")
print(f"  {len(state_fips)} states × {len(sectors)} sectors = {len(state_fips)*len(sectors)} series")
print("=" * 60)

smu_series = []
smu_meta   = {}
for fips, state_name in zip(state_fips, state_names):
    for sector_code, sector_name in sectors.items():
        # ORRECT FORMAT: append '0' to 2-digit code → 3-digit sector code
        # '06'→'060', '50'→'500', '55'→'550', '90'→'900'
        three = sector_code + '0'
        sid   = f"SMU{fips}00000{three}0000001"
        smu_series.append(sid)
        smu_meta[sid] = {"State": state_name, "Sector": sector_name}

# Print sample IDs so you can visually verify
print("  Sample series IDs:")
for sid in list(smu_meta.keys())[:6]:
    print(f"    {sid} → {smu_meta[sid]['Sector']} / {smu_meta[sid]['State']}")

smu_records   = []
total_batches = len(list(chunked(smu_series, BATCH_SIZE))) * len(TIME_PERIODS)
batch_count   = 0

for start, end in TIME_PERIODS:
    for batch in chunked(smu_series, BATCH_SIZE):
        batch_count += 1
        print(f"  Batch {batch_count}/{total_batches} | {len(batch)} series | {start}–{end}")
        result = fetch_bls_v2(batch, start, end)
        smu_records.extend(parse_bls_response(result, smu_meta))
        time.sleep(1)

    # Checkpoint after each time period
    pd.DataFrame(smu_records).to_csv(f"checkpoint_smu_{start}_{end}.csv", index=False)
    print(f"  ✓ Checkpoint saved: checkpoint_smu_{start}_{end}.csv\n")

df_smu = pd.DataFrame(smu_records)
print(f"  ✓ {len(df_smu):,} SMU records fetched")

# ── STEP 3: PIVOT SECTORS INTO COLUMNS ───────────────────────────────────────

print("\n" + "=" * 60)
print("STEP 3: Pivoting sectors into columns...")
print("=" * 60)

if df_smu.empty:
    raise ValueError("No SMU records fetched — check API key and series IDs")

df_sectors = df_smu.pivot_table(
    index   = ["State", "Year", "Month"],
    columns = "Sector",
    values  = "Value",
    aggfunc = "first",
).reset_index()
df_sectors.columns.name = None
print(f"  ✓ Pivoted shape: {df_sectors.shape}")

# ── STEP 4: MERGE ─────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("STEP 4: Merging unemployment + sectors...")
print("=" * 60)

df_merged = df_laus.merge(df_sectors, on=["State", "Year", "Month"], how="left")

df_merged["Date"] = pd.to_datetime(
    df_merged[["Year", "Month"]].rename(
        columns={"Year": "year", "Month": "month"}
    ).assign(day=1)
)

df_merged["Total_Tech_Employment_Thousands"] = (
    df_merged.get("Information",                    pd.Series(0, index=df_merged.index)).fillna(0) +
    df_merged.get("Professional_and_Tech_Services", pd.Series(0, index=df_merged.index)).fillna(0)
)

col_order = (
    ["State", "Date", "Year", "Month", "Unemployment_Rate"]
    + [c for c in sectors.values() if c in df_merged.columns]
    + ["Total_Tech_Employment_Thousands"]
)
df_merged = df_merged[col_order].sort_values(["State", "Year", "Month"]).reset_index(drop=True)

# ── STEP 5: VALIDATE ──────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("FINAL VALIDATION")
print("=" * 60)
print(f"  Total records : {len(df_merged):,}")
print(f"  States        : {df_merged['State'].nunique()}")
print(f"  Columns       : {len(df_merged.columns)}")
print(f"  Date range    : {df_merged['Date'].min().strftime('%b %Y')} → {df_merged['Date'].max().strftime('%b %Y')}")

print(f"\n  Missing values:")
print(df_merged.isna().sum().to_string())

# Spot check Alabama Jan 2016
print(f"\n  Alabama Jan 2016 spot check:")
al = df_merged[(df_merged['State']=='Alabama') &
               (df_merged['Year']==2016) &
               (df_merged['Month']==1)]
for col in ['Information','Professional_and_Tech_Services',
            'Leisure_and_Hospitality','Total_Tech_Employment_Thousands']:
    if col in al.columns:
        print(f"    {col:40s}: {al[col].values[0]:.1f}")

print(f"\n  Expected Alabama Jan 2016 (approximate):")
print(f"    Information                              : 80–110")
print(f"    Professional_and_Tech_Services           : 180–220")
print(f"    Leisure_and_Hospitality                  : 190–230")
print(f"    Total_Tech_Employment_Thousands          : 260–330")

# ── STEP 6: EXPORT ────────────────────────────────────────────────────────────

out_file = "state_unemployment_all_sectors_2016_2026_FINAL.csv"
df_merged.to_csv(out_file, index=False)
print(f"\nSaved: {out_file}")
print("Upload this file to Claude for the dashboard!")

In [2]:
df1 = pd.read_csv("state_unemployment_all_sectors_2016_2026_FINAL.csv")
df1.isna().sum()

State                                0
Date                                 0
Year                                 0
Month                                0
Unemployment_Rate                   50
Goods_Producing                      0
Service_Providing                    0
Mining_and_Logging                 244
Construction                       244
Manufacturing                        0
Trade_Transport_and_Utilities        0
Information                          0
Financial_Activities                 0
Professional_and_Tech_Services       0
Education_and_Health_Services        0
Leisure_and_Hospitality              0
Government                           0
Total_Tech_Employment_Thousands      0
dtype: int64

In [3]:
alabama = df1[df1['State'] == 'Alabama']
alabama.head()

,State,Date,Year,Month,Unemployment_Rate,Goods_Producing,Service_Providing,Mining_and_Logging,Construction,Manufacturing,Trade_Transport_and_Utilities,Information,Financial_Activities,Professional_and_Tech_Services,Education_and_Health_Services,Leisure_and_Hospitality,Government,Total_Tech_Employment_Thousands
0,Alabama,2016-01-01,2016,1,5.9,351.2,1611.6,9.9,82.7,258.6,376.4,20.8,95.1,229.1,233.2,186.2,378.5,249.9
1,Alabama,2016-02-01,2016,2,5.9,351.2,1623.6,9.5,82.7,259.0,375.8,20.7,95.3,230.7,236.7,190.0,381.7,251.4
2,Alabama,2016-03-01,2016,3,5.9,352.8,1632.0,9.5,84.5,258.8,377.3,20.6,95.5,231.9,235.6,194.6,383.0,252.5
3,Alabama,2016-04-01,2016,4,5.8,354.6,1646.8,9.3,85.4,259.9,379.2,20.8,95.9,235.4,237.9,199.7,383.5,256.2
4,Alabama,2016-05-01,2016,5,5.8,353.6,1650.0,9.2,84.5,259.9,379.7,21.0,96.1,234.9,237.3,202.2,384.6,255.9


In [4]:
last_year = df1['Year'].max()
data = df1[df1['Year'] == last_year]
last_month = data['Month'].max()
data = data[data['Month'] == last_month]

In [ ]:
# data.head()

,State,Date,Year,Month,Unemployment_Rate,Goods_Producing,Service_Providing,Mining_and_Logging,Construction,Manufacturing,Trade_Transport_and_Utilities,Information,Financial_Activities,Professional_and_Tech_Services,Education_and_Health_Services,Leisure_and_Hospitality,Government,Total_Tech_Employment_Thousands
121,Alabama,2026-02-01,2026,2,2.7,403.1,1792.2,9.3,110.5,283.3,400.0,21.9,102.8,264.3,266.3,213.0,429.8,286.2
243,Alaska,2026-02-01,2026,2,4.7,42.5,284.4,13.7,16.7,12.1,64.3,4.1,10.9,28.5,54.1,31.9,78.7,32.6
365,Arizona,2026-02-01,2026,2,4.6,432.8,2853.9,16.6,223.4,192.8,621.0,48.9,241.3,464.2,562.5,372.4,438.2,513.1
487,Arkansas,2026-02-01,2026,2,4.4,228.2,1108.9,4.8,65.0,158.4,272.1,11.7,58.8,165.0,212.8,129.8,206.5,176.7
609,California,2026-02-01,2026,2,5.4,2089.7,15939.9,18.0,867.5,1204.2,3028.3,521.2,786.3,2752.4,3558.8,2016.1,2677.2,3273.6


In [6]:
data = data[data['State'] == 'Alabama']
data = data['Goods_Producing']

In [7]:

data.head()

121    403.1
Name: Goods_Producing, dtype: float64

In [ ]:
# df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 6100 entries, 0 to 6099
Data columns (total 18 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   State                            6100 non-null   str    
 1   Date                             6100 non-null   str    
 2   Year                             6100 non-null   int64  
 3   Month                            6100 non-null   int64  
 4   Unemployment_Rate                6050 non-null   float64
 5   Goods_Producing                  6100 non-null   float64
 6   Service_Providing                6100 non-null   float64
 7   Mining_and_Logging               5856 non-null   float64
 8   Construction                     5856 non-null   float64
 9   Manufacturing                    6100 non-null   float64
 10  Trade_Transport_and_Utilities    6100 non-null   float64
 11  Information                      6100 non-null   float64
 12  Financial_Activities           